In [1]:
from itertools import product
import shutil

In [2]:
#arguments
#expect: one starting config file for each wd_pt model


# ### FT sweep 1 -- llama-1B-20BT on simplescaling
# model_size = 'llama-1B-20BT'
# wd_pt_lst = [0.1, 0.5, 1.0]
# sft_dataset = 'simplescaling'


# ### FT sweep 2 -- olmo-1B-30BT on metamathqa
# model_size = 'olmo-1B-30BT'
# wd_pt_lst = [0.1, 0.3, 1.0] #[0.1, 0.3, 0.6, 1.0]
# sft_dataset = 'metamathqa'


### FT sweep 3 -- olmo-1B-30BT on simplescaling
model_size = 'olmo-1B-30BT'
wd_pt_lst = [0.1, 0.3, 1.0] #[0.1, 0.3, 0.6, 1.0]
sft_dataset = 'simplescaling'


#default hyperparams
default = ('1.0e-5', 16, 0.0) #lr, bs, wd_ft

In [3]:
#hyperparamter values to sweep: learning rate, batch size, weight decay during finetuning
lr_lst = ['6.0e-5'] #['1.0e-5', '3.0e-5', '6.0e-5']
bs_lst = [8, 16, 32] #actual batch size = batch_size * 4 GPUs = 32, 64, 128
wd_ft_lst = [0.0, 0.1, 1.0]

In [4]:
#create hyperparam combos
combos = list(product(lr_lst, bs_lst, wd_ft_lst))

#remove default combo from combos
combos = [combo for combo in combos if combo != default]

### checks
lr_default, bs_default, wd_ft_default = default
if lr_default in lr_lst and bs_default in bs_lst and wd_ft_default in wd_ft_lst:
    assert len(combos) == len(lr_lst) * len(bs_lst) * len(wd_ft_lst) - 1 #check combo length
else:
    assert len(combos) == len(lr_lst) * len(bs_lst) * len(wd_ft_lst) #check combo length
assert default not in combos #check that default is not in combos

In [5]:
n_newconfig_files_created = 0

#make a copy of config file for metamathqa
print(f"Creating config file for: model {model_size}, sft_dataset {sft_dataset}")

for wd_pt in wd_pt_lst:
    print(f'wd_pretrain {wd_pt}')

    for i, combo in enumerate(combos):
        lr, bs, wd_ft = combo
        print(f'   {i+1}. lr = {lr}, bs = {bs}, wd_ft = {wd_ft}')

        ### copy config file
        if model_size in ['olmo-1B-210BT', 'olmo-1B-30BT']:
            og_config_file_path = f'config_hub/custom_configs/sweep_hyperparams/ft_{sft_dataset}/{model_size}-weightdecay{wd_pt}-{sft_dataset}.yaml'
            new_config_file_path = f'config_hub/custom_configs/sweep_hyperparams/ft_{sft_dataset}/{model_size}-weightdecay{wd_pt}-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}.yaml'
        else:
            og_config_file_path = f'config_hub/custom_configs/sweep_hyperparams/ft_{sft_dataset}/{model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}.yaml'
            new_config_file_path = f'config_hub/custom_configs/sweep_hyperparams/ft_{sft_dataset}/{model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}.yaml'
        shutil.copyfile(og_config_file_path, new_config_file_path)

        ### make edits to the new config file
        with open(new_config_file_path, "r", encoding="utf-8") as f:
            text = f.read()

            #changes these variables in config file:
                # output_dir: llamafactory_out/llama-1B-20BT-weightdecay0.1-seed42-simplescaling
                # per_device_train_batch_size: 16
                # learning_rate: 1.0e-5
                # weight_decay: 0.0
                # run_name: llama-1B-20BT-weightdecay0.1-seed42-simplescaling

            #change output_dir
            if model_size in ['olmo-1B-210BT', 'olmo-1B-30BT']:
                old_str = f"output_dir: llamafactory_out/{model_size}-weightdecay{wd_pt}-{sft_dataset}"
                new_str = f"output_dir: /n/netscratch/doshi-velez_lab/Everyone/models/sweep/{model_size}-weightdecay{wd_pt}-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}"
            else:
                old_str = f"output_dir: llamafactory_out/{model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}"
                new_str = f"output_dir: /n/netscratch/doshi-velez_lab/Everyone/models/sweep/{model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}"
            text = text.replace(old_str, new_str)

            #change per_device_train_batch_size
            old_str = "per_device_train_batch_size: 16"
            new_str = f"per_device_train_batch_size: {bs}"
            text = text.replace(old_str, new_str)

            #change learning_rate
            old_str = "learning_rate: 1.0e-5"
            new_str = f"learning_rate: {lr}"
            text = text.replace(old_str, new_str)

            #change weight_decay
            old_str = "weight_decay: 0.0"
            new_str = f"weight_decay: {wd_ft}"
            text = text.replace(old_str, new_str)

            #change run_name
            if model_size in ['olmo-1B-210BT', 'olmo-1B-30BT']:
                old_str = f"run_name: {model_size}-weightdecay{wd_pt}-{sft_dataset}"
                new_str = f"run_name: {model_size}-weightdecay{wd_pt}-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}"
            else:
                old_str = f"run_name: {model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}"
                new_str = f"run_name: {model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}"
            text = text.replace(old_str, new_str)
            
            #write new config file
            with open(new_config_file_path, "w", encoding="utf-8") as f:
                f.write(text)

        n_newconfig_files_created += 1

print('# new config files created: ', n_newconfig_files_created)
print("Complete!")

Creating config file for: model olmo-1B-30BT, sft_dataset simplescaling
wd_pretrain 0.1
   1. lr = 6.0e-5, bs = 8, wd_ft = 0.0
   2. lr = 6.0e-5, bs = 8, wd_ft = 0.1
   3. lr = 6.0e-5, bs = 8, wd_ft = 1.0
   4. lr = 6.0e-5, bs = 16, wd_ft = 0.0
   5. lr = 6.0e-5, bs = 16, wd_ft = 0.1
   6. lr = 6.0e-5, bs = 16, wd_ft = 1.0
   7. lr = 6.0e-5, bs = 32, wd_ft = 0.0
   8. lr = 6.0e-5, bs = 32, wd_ft = 0.1
   9. lr = 6.0e-5, bs = 32, wd_ft = 1.0
wd_pretrain 0.3
   1. lr = 6.0e-5, bs = 8, wd_ft = 0.0
   2. lr = 6.0e-5, bs = 8, wd_ft = 0.1
   3. lr = 6.0e-5, bs = 8, wd_ft = 1.0
   4. lr = 6.0e-5, bs = 16, wd_ft = 0.0
   5. lr = 6.0e-5, bs = 16, wd_ft = 0.1
   6. lr = 6.0e-5, bs = 16, wd_ft = 1.0
   7. lr = 6.0e-5, bs = 32, wd_ft = 0.0
   8. lr = 6.0e-5, bs = 32, wd_ft = 0.1
   9. lr = 6.0e-5, bs = 32, wd_ft = 1.0
wd_pretrain 1.0
   1. lr = 6.0e-5, bs = 8, wd_ft = 0.0
   2. lr = 6.0e-5, bs = 8, wd_ft = 0.1
   3. lr = 6.0e-5, bs = 8, wd_ft = 1.0
   4. lr = 6.0e-5, bs = 16, wd_ft = 0.0
   5. lr 

In [6]:
#for olmo-1B models, for the bs=32 setting

#change config file 
     # from bs=32 and gradient_accumulation_steps=1 -- this causes OOM errors
     # to bs=16 and gradient_accumulation_steps=2

if model_size in ['olmo-1B-210BT', 'olmo-1B-30BT']:
        
    ### get all combos with bs=32
    combos_bs32 = []

    for combo in combos:
        if combo[1] == 32:
            combos_bs32.append(combo)

    print("combos with bs=32, to be edited:")
    print(combos_bs32)

    ### edit config files for bs=32
    n_config_files_edited = 0

    for wd_pt in wd_pt_lst:
        print(f'wd_pretrain {wd_pt}')

        for j, combo_bs32 in enumerate(combos_bs32):
            # print(f'{j+1}. {combo_bs32}')

            lr, bs, wd_ft = combo_bs32
            print(f'   {j+1}. lr = {lr}, bs = {bs}, wd_ft = {wd_ft}')

            ### edit config file created in previous cell
            if model_size in ['olmo-1B-210BT', 'olmo-1B-30BT']:
                new_config_file_path = f'config_hub/custom_configs/sweep_hyperparams/ft_{sft_dataset}/{model_size}-weightdecay{wd_pt}-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}.yaml'
            else:
                new_config_file_path = f'config_hub/custom_configs/sweep_hyperparams/ft_{sft_dataset}/{model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}.yaml'

            with open(new_config_file_path, "r", encoding="utf-8") as f:
                text = f.read()

                #change per_device_train_batch_size
                old_str = "per_device_train_batch_size: 32"
                new_str = f"per_device_train_batch_size: 16"
                text = text.replace(old_str, new_str)

                #change gradient_accumulation_steps
                old_str = "gradient_accumulation_steps: 1"
                new_str = f"gradient_accumulation_steps: 2"
                text = text.replace(old_str, new_str)

                #overwrite config file
                with open(new_config_file_path, "w", encoding="utf-8") as f:
                    f.write(text)
                
            n_config_files_edited += 1

    print('# config files edited: ', n_config_files_edited)
    print("Complete!")


combos with bs=32, to be edited:
[('6.0e-5', 32, 0.0), ('6.0e-5', 32, 0.1), ('6.0e-5', 32, 1.0)]
wd_pretrain 0.1
   1. lr = 6.0e-5, bs = 32, wd_ft = 0.0
   2. lr = 6.0e-5, bs = 32, wd_ft = 0.1
   3. lr = 6.0e-5, bs = 32, wd_ft = 1.0
wd_pretrain 0.3
   1. lr = 6.0e-5, bs = 32, wd_ft = 0.0
   2. lr = 6.0e-5, bs = 32, wd_ft = 0.1
   3. lr = 6.0e-5, bs = 32, wd_ft = 1.0
wd_pretrain 1.0
   1. lr = 6.0e-5, bs = 32, wd_ft = 0.0
   2. lr = 6.0e-5, bs = 32, wd_ft = 0.1
   3. lr = 6.0e-5, bs = 32, wd_ft = 1.0
# config files edited:  9
Complete!
